# `cfdtools.post` examples

Examples of reading an OpenFOAM case and plotting it with `cfdtools.post`, using an
`annulus-2D` case (2D Couette flow between a rotating bob and a stationary cup).

Requirements:

- A case that has been run, e.g. `cases/smoke`:
  `pyFoamPrepareCase.py cases/smoke --clone-case=base/annulus-2D`, then `pimpleFoam` in it.
- A Jupyter kernel in the `cfd` environment: `pip install -e ".[notebook]"` from the repository root.

Set `CASE` below to the case to plot. The path is relative to this notebook (`post/`).

In [ ]:
%matplotlib inline
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from cfdtools import post
from cfdtools.monitor import find_series, read_dat

CASE = Path("../cases/smoke")

## 1. Read a case

`read_case` returns the mesh at one saved time (default: the latest) and the time it read.
Fields are stored per cell in `mesh.cell_data`.

In [ ]:
mesh, t = post.read_case(CASE)
print(f"time {t} s, {mesh.n_cells} cells, fields: {list(mesh.cell_data.keys())}")
print("bounds [m]:", np.round(mesh.bounds, 6))

## 2. Parameters the case was built with

`case_parameters` reads `PyFoamPrepareCaseParameters`: the values from `default.parameters`,
any overrides, and the SI values computed in `derivedParameters.py`.

In [ ]:
p = post.case_parameters(CASE)
Ri, Ro, ecc, omega, nu = (float(p[k]) for k in ("Ri", "Ro", "ecc", "omega", "nu"))
print(f"Ri = {Ri*1e3} mm, Ro = {Ro*1e3} mm, eccentricity = {ecc*1e3} mm")
print(f"omega = {omega:.3f} rad/s, nu = {nu:.3e} m^2/s")

## 3. Figure style

`use_style()` sets matplotlib defaults for print: white background, thin lines, muted axes,
fonts embedded as text in PDFs. Call it once before plotting.

In [ ]:
post.use_style()

## 4. A field on the mesh

`plot_field` draws each cell as a polygon coloured by its value, so the plot shows the real
cells without interpolation. For a vector field, `which` picks the magnitude (default) or a
component. The returned object is passed to `colorbar`.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
pc = post.plot_field(ax, mesh, "U")
fig.colorbar(pc, ax=ax, label="|U| [m/s]", shrink=0.8)
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
plt.show()

A signed quantity, such as one velocity component, reads best with a diverging colour map
centred on zero. `cmap`, `vmin` and `vmax` set the colours and range.

In [ ]:
ux = post.component(mesh.cell_data["U"], "x")
lim = abs(ux).max()

fig, ax = plt.subplots(figsize=(4, 4))
pc = post.plot_field(ax, mesh, "U", which="x", cmap="coolwarm", vmin=-lim, vmax=lim)
fig.colorbar(pc, ax=ax, label="$U_x$ [m/s]", shrink=0.8)
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
plt.show()

## 5. Unrolling the annulus

The gap (0.5 mm) is thin compared with the radius (10 mm), so the full view shows a thin ring.
`transform` maps each cell's vertices to new coordinates. Here: angle about the cup axis
against radius.

In [ ]:
def unroll(poly):
    """One cell's (x, y) vertices -> (angle in degrees, radius in mm)."""
    theta = np.degrees(np.unwrap(np.arctan2(poly[:, 1], poly[:, 0])))
    if theta.mean() > 180:
        theta -= 360
    elif theta.mean() < -180:
        theta += 360
    return np.column_stack([theta, np.hypot(poly[:, 0], poly[:, 1]) * 1e3])

fig, ax = plt.subplots(figsize=(6, 2.5))
pc = post.plot_field(ax, mesh, "p", transform=unroll)
fig.colorbar(pc, ax=ax, label="p [m$^2$/s$^2$]")
ax.set_xlim(-180, 180)
ax.set_ylim((Ri - ecc) * 1e3, Ro * 1e3)
ax.set_xlabel(r"$\theta$ [deg]")
ax.set_ylabel("r [mm]")
plt.show()

## 6. Values along a line

`sample_line` samples a field along a straight line, interpolating linearly between cells.
Here the line runs across the gap along the +x axis, through the middle of the mesh thickness,
and is compared with the analytical Couette profile (valid for the concentric case).

In [ ]:
z = 0.5 * (mesh.bounds[4] + mesh.bounds[5])
start, end = (ecc + Ri, 0.0, z), (Ro, 0.0, z)
d, uy = post.sample_line(mesh, start, end, "U", which="y")

r = Ri + np.linspace(0, Ro - Ri, 200)
u_exact = omega * Ri**2 / r * (Ro**2 - r**2) / (Ro**2 - Ri**2)

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot((r - Ri) * 1e3, u_exact, color=post.INK, linestyle="--", linewidth=1, label="Analytical")
ax.plot(d * 1e3, uy, color=post.SERIES[0], label="OpenFOAM (sample_line)")
ax.set_xlabel("distance from bob [mm]")
ax.set_ylabel(r"$u_\theta$ [m/s]")
ax.legend()
plt.show()

## 7. Working with cell values directly

For exact cell values (no interpolation), use `cell_centres` and `mesh.cell_data`. Here the
tangential velocity of every cell is plotted against its radius. `cell_vertices` gives each
cell's corners; the radius is taken midway between the inner and outer corners, because the
centroid of a flat-sided cell lies slightly inside the curved arc.

In [ ]:
c = post.cell_centres(mesh)
U = mesh.cell_data["U"]
u_theta = (-c[:, 1] * U[:, 0] + c[:, 0] * U[:, 1]) / np.hypot(c[:, 0], c[:, 1])
r_cell = np.array([0.5 * (np.hypot(v[:, 0], v[:, 1]).min() + np.hypot(v[:, 0], v[:, 1]).max())
                   for v in post.cell_vertices(mesh)])

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot((r - Ri) * 1e3, u_exact, color=post.INK, linestyle="--", linewidth=1, label="Analytical")
ax.plot((r_cell - Ri) * 1e3, u_theta, linestyle="none", marker="o", markersize=3,
        color=post.SERIES[0], label="OpenFOAM (cells)")
ax.set_xlabel("distance from bob [mm]")
ax.set_ylabel(r"$u_\theta$ [m/s]")
ax.legend()
plt.show()

## 8. Several saved times

`read_case(case, time)` reads the saved time nearest to `time`. Plotting the profile at
several times shows the flow developing after the bob starts rotating.

In [ ]:
times = [0.01, 0.02, 0.04, 0.08, 0.2]
colours = post.sequential_cmap()(np.linspace(0.25, 1.0, len(times)))

fig, ax = plt.subplots(figsize=(4, 3))
for time, colour in zip(times, colours):
    m, t_read = post.read_case(CASE, time)
    d, uy = post.sample_line(m, start, end, "U", which="y")
    ax.plot(d * 1e3, uy, color=colour, label=f"t = {t_read:g} s")
ax.set_xlabel("distance from bob [mm]")
ax.set_ylabel(r"$u_\theta$ [m/s]")
ax.legend()
plt.show()

## 9. Time histories from `postProcessing/`

`find_series` and `read_dat` (from `cfdtools.monitor`) read the function-object output of the
latest run. Here: the torque on the bob. The mesh is one cell thick, so the torque is divided
by the thickness to give torque per metre of bob length.

In [ ]:
series = find_series(CASE)
print("available:", list(series))

moment = read_dat(series["bobTorque/moment"])
thickness = mesh.bounds[5] - mesh.bounds[4]
torque_per_m = np.array(moment["total_z"]) / thickness

# Analytical torque per unit length, concentric cylinders.
mu = float(p["mu_mPa_s"]) * 1e-3
T_exact = -4 * np.pi * mu * omega * Ri**2 * Ro**2 / (Ro**2 - Ri**2)

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(moment["Time"], torque_per_m * 1e3, color=post.SERIES[0], label="OpenFOAM")
ax.axhline(T_exact * 1e3, color=post.INK, linestyle="--", linewidth=1, label="Analytical")
ax.set_xlabel("t [s]")
ax.set_ylabel("torque per length [mN m/m]")
ax.legend()
plt.show()
print(f"final: {torque_per_m[-1]:.4e} N m/m, analytical: {T_exact:.4e} N m/m")

## 10. Saving a figure

Save as PDF for papers (vector text and axes; the cells are embedded as an image when
`plot_field(..., rasterized=True)`, the default) and PNG for slides.

In [ ]:
out = CASE / "figures"
out.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(3.5, 3.5))
pc = post.plot_field(ax, mesh, "U")
fig.colorbar(pc, ax=ax, label="|U| [m/s]", shrink=0.8)
fig.savefig(out / "U_magnitude.pdf")
fig.savefig(out / "U_magnitude.png")
plt.close(fig)
print("saved to", out)